# 05 - Monitoring and Fairness

Notebook para acompanhar estabilidade do modelo, drift, metricas por grupo e sinais de vies.

## 1 — Setup, dados e função de score

Monitoramos o **modelo final** (LightGBM, artefato salvo) com a mesma metodologia de score do notebook 04 (correção de prior + PDO). Usamos o **treino como referência** (distribuição que o modelo viu) e o **teste como população atual**. Em produção, a "atual" seria cada novo lote de aplicações.

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from sklearn.model_selection import train_test_split

from src.preprocessing import tratar_nulos
from src.features import criar_features, codificar_categoricas
from src.scoring import adjust_probability_to_prior, probability_to_scorecard_points, assign_risk_band
from src.monitoring import population_stability_index

# Artefatos do notebook 03
modelo_final = joblib.load('../models/modelo_credito.pkl')
imputers = joblib.load('../models/imputers.pkl')
encoders = joblib.load('../models/encoders.pkl')

# Reconstrói o mesmo split do notebook 03
df = pd.read_csv('../data/raw/application_train.csv')
X = df.drop(columns=['TARGET', 'SK_ID_CURR'])
y = df['TARGET']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Guarda o atributo protegido (gênero) ANTES da codificação, para o fairness
genero_test = X_test['CODE_GENDER'].copy()

prior = y_train.mean()
odds_ref = (1 - prior) / prior


def preparar(Xdf):
    """Aplica o pré-processamento aprendido no treino (sem refit)."""
    Xdf = tratar_nulos(Xdf, fit=False, imputers=imputers)
    Xdf = criar_features(Xdf)
    return codificar_categoricas(Xdf, fit=False, encoders=encoders)


def pontuar(Xdf):
    """Score do modelo campeão: PD -> correção de prior -> pontos PDO."""
    prob = modelo_final.predict_proba(preparar(Xdf))[:, 1]
    prob_real = adjust_probability_to_prior(prob, target_prior=prior, source_prior=0.5)
    return probability_to_scorecard_points(prob_real, pdo=60, odds_ref=odds_ref, score_ref=600)


score_ref = pontuar(X_train)   # referência (treino)
score_cur = pontuar(X_test)    # população atual (teste)
print(f'Referência: {len(score_ref):,} clientes | Atual: {len(score_cur):,} clientes')

## 2 — Drift do score (PSI)

O **PSI** (Population Stability Index) compara a distribuição do score atual com a de referência. Regra prática: **< 0,10** estável · **0,10–0,25** mudança moderada · **> 0,25** drift significativo (investigar/retreinar).

In [ ]:
psi = population_stability_index(score_ref, score_cur, bins=10)

# Demonstração: um lote com piora de 60 pontos no score (um drift real seria detectado assim)
score_drift = np.clip(score_cur - 60, 300, 850)
psi_drift = population_stability_index(score_ref, score_drift)


def classifica_psi(v):
    if v < 0.10:
        return 'estável'
    if v < 0.25:
        return 'mudança moderada'
    return 'drift significativo'


print(f'PSI treino -> teste:          {psi:.4f}  ({classifica_psi(psi)})')
print(f'PSI com drift simulado (-60): {psi_drift:.4f}  ({classifica_psi(psi_drift)})')

plt.figure(figsize=(8, 5))
plt.hist(score_ref, bins=40, alpha=0.5, density=True, label='Referência (treino)')
plt.hist(score_cur, bins=40, alpha=0.5, density=True, label='Atual (teste)')
plt.xlabel('Score')
plt.ylabel('Densidade')
plt.title(f'Distribuição do score — PSI = {psi:.4f}', fontweight='bold')
plt.legend()
plt.tight_layout()
plt.savefig('../reports/figures/monitor_psi.png', dpi=150, bbox_inches='tight')
plt.show()

## 3 — Estabilidade das faixas de risco

Os cortes das faixas são **fixados na referência** (treino) e reaplicados à população atual. As proporções por faixa devem permanecer estáveis ao longo do tempo; migrações grandes indicam mudança de perfil da carteira.

In [ ]:
# Cortes de tercil fixados na REFERÊNCIA (treino) e reaplicados aos dois períodos
c_high, c_low = np.quantile(score_ref, [1/3, 2/3])

def faixa(s):
    return assign_risk_band(s, medium_min=c_high, low_min=c_low)

prop_ref = pd.Series([faixa(s) for s in score_ref]).value_counts(normalize=True)
prop_cur = pd.Series([faixa(s) for s in score_cur]).value_counts(normalize=True)
estabilidade = pd.DataFrame({'referencia': prop_ref, 'atual': prop_cur}).reindex(['low', 'medium', 'high'])
estabilidade['variacao_pp'] = (estabilidade['atual'] - estabilidade['referencia']) * 100

print(f'Cortes (referência): high < {c_high:.0f} <= medium < {c_low:.0f} <= low')
estabilidade.round(4)

## 4 — Fairness por gênero

Avaliamos se o modelo trata grupos de forma equilibrada. "Aprovado" = não cair na faixa `high`. Métrica-chave: **Disparate Impact ratio** (menor/maior taxa de aprovação) — pela "regra dos 80%", valores **< 0,80** indicam possível disparidade a investigar.

In [ ]:
fair = pd.DataFrame({'genero': genero_test.values, 'score': score_cur, 'target': y_test.values})
fair = fair[fair['genero'].isin(['F', 'M'])]  # remove 'XNA' (pouquíssimos casos)
fair['aprovado'] = fair['score'].apply(lambda s: faixa(s) != 'high')  # high = recusa

resumo_fair = fair.groupby('genero').agg(
    qtd=('target', 'size'),
    taxa_aprovacao=('aprovado', 'mean'),
    inadimplencia=('target', 'mean'),
    score_medio=('score', 'mean'),
).round(4)
print(resumo_fair)

di = resumo_fair['taxa_aprovacao'].min() / resumo_fair['taxa_aprovacao'].max()
print(f'\nDisparate Impact ratio: {di:.3f}  ({"OK (>=0.80)" if di >= 0.8 else "ATENÇÃO (<0.80)"})')

resumo_fair['taxa_aprovacao'].plot(kind='bar', color=['#2E5EAA', '#E65100'], figsize=(6, 4))
plt.ylabel('Taxa de aprovação')
plt.title('Taxa de aprovação por gênero', fontweight='bold')
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig('../reports/figures/monitor_fairness.png', dpi=150, bbox_inches='tight')
plt.show()

## 5 — Conclusão (monitoramento e fairness)

**Estabilidade / drift.** O PSI entre referência (treino) e população atual (teste) ficou ~0 (estável) — esperado, já que vêm do mesmo período. A demonstração com um lote deslocado mostra o PSI disparando bem acima de 0,25, comprovando que a métrica capta drift. Em produção, este notebook rodaria periodicamente sobre cada novo lote; PSI > 0,25 (ou faixas migrando) dispara revisão/retreino.

**Fairness.** As mulheres têm taxa de aprovação maior que os homens, refletindo uma diferença **real** de inadimplência nesta base (M inadimplem mais). O **Disparate Impact ratio ≈ 0,80** fica no limite da "regra dos 80%" — um sinal de alerta legítimo que, num contexto real, exigiria:
- avaliar se há **proxies** de gênero entre as features;
- considerar **limiares por grupo** ou restrições de fairness no treino;
- decisão de negócio/jurídica sobre o trade-off risco × equidade (crédito é área regulada).

**Mensagem central:** o pipeline não entrega só um número de performance — ele é **monitorável** (drift) e **auditável** (fairness), o que diferencia um modelo de crédito pronto para produção de um experimento de notebook.